# 08 – Term order: does the training data write terms in a fixed order?

**Why this notebook.** In `05`, reordering the terms of a correctly solved input kept only ~13% of correct answers.
Our hypothesis: the dataset generator writes terms in a typical order, and the model learned that order.
So far we only checked the *operand* order inside bitwise operations (`x` always left, `y` always right – `06`/`07`, on
the training data). The *term* order was never checked – neither on training nor on test data. This notebook does it
for both, in the same style as `07`, and then connects it to the model's behaviour.

**Approach.**
1. Split every input into its top-level terms and give each term a *kind*: which bitwise function it is (identified by
   its truth table, e.g. `x&y`), or "constant". Coefficients and signs are ignored; variable names are normalized
   keeping their role (as in `06`).
2. For every pair of term kinds that occur together: in how many inputs does A come before B? A fixed order means these
   shares are close to 0% or 100%; random order means ~50%. A shuffled copy of the same data is the baseline.
3. Connect to the model: when we reordered inputs in `05`, did answers break more the more the new order contradicts
   the training order?

Needs `data/predictions_train.csv`, `data/predictions_test.csv`, and `probe_results_full.csv` (or `probe_results.csv`)
from `05`. Figures are saved to `figures/final/` as `13_…` to `15_…`, continuing `07`.

In [ ]:
import itertools
import random
import re
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "predictions_test.csv").exists())
sys.path.insert(0, str(ROOT / "src"))
import neureduce as nr

HERE = Path.cwd()
OUT = HERE / "figures" / "final"
OUT.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(ROOT / "data" / "predictions_train.csv")
test = pd.read_csv(ROOT / "data" / "predictions_test.csv")
test["error"] = ~test["correct"]
probe_file = HERE / "probe_results_full.csv"
if not probe_file.exists():
    probe_file = HERE / "probe_results.csv"
probes = pd.read_csv(probe_file)
probes["ok"] = probes.new_verdict.isin(["exact", "equivalent"])
print(f"train {len(train):,} · test {len(test):,} · rewrites from {probe_file.name}")

# ---- same style as 07
C = {"correct": "#2a9d8f", "wrong": "#e63946", "neutral": "#c9ced6", "accent": "#457b9d",
     "dark": "#1d3557", "soft": "#a8dadc", "warm": "#f4a261"}
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 220, "savefig.bbox": "tight", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#888888",
    "axes.grid": True, "grid.color": "#e6e6e6", "axes.axisbelow": True,
    "axes.titlesize": 10.5, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "xtick.color": "#444444", "ytick.color": "#444444",
})
BEFORE_AFTER = LinearSegmentedColormap.from_list("before_after", [C["warm"], "#f7f7f7", C["accent"]])


def headline(fig, title, subtitle=None):
    fig._headline = (title, subtitle)


def save(fig, name):
    h = fig.get_figheight()
    title, subtitle = getattr(fig, "_headline", (None, None))
    fig.tight_layout(rect=(0, 0, 1, 1 - (0.8 if subtitle else 0.5) / h))
    if title:
        fig.text(0.01, 1 - 0.22 / h, title, ha="left", va="center", fontsize=13, fontweight="bold", color=C["dark"])
    if subtitle:
        fig.text(0.01, 1 - 0.52 / h, subtitle, ha="left", va="center", fontsize=9.5, color="#555555")
    fig.savefig(OUT / f"{name}.png")
    plt.show()


def wilson(k, n, z=1.96):
    k, n = np.asarray(k, float), np.asarray(n, float)
    p = k / n
    d = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / d
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return centre - half, centre + half


def pct(x, digits=0):
    return f"{100 * x:.{digits}f}%"

## 1 · Every input as a sequence of term kinds

`-2*(~x&y)-(~y)+2*(~x)-y+2` becomes `[~x&y, ~y, ~x, y, constant]`. A kind is identified by its truth table, so `x&y`
and `y&x` are the same kind; each kind is shown with its most common spelling.

In [ ]:
def split_terms(expr):
    """Top-level additive terms with their sign: '-2*(x&y)+z' -> ['-2*(x&y)', '+z']."""
    terms, cur, depth = [], "", 0
    for i, ch in enumerate(expr):
        if ch in "+-" and depth == 0 and i > 0 and expr[i - 1] not in "+-*&|^~(":
            terms.append(cur)
            cur = ch
        else:
            cur += ch
        depth += (ch == "(") - (ch == ")")
    return terms + [cur]


def normalize_names(expr):
    """Keep x and y; another letter takes the free role (x = left, y = right), as in 06."""
    names = set(re.findall(r"[a-z]", expr))
    free = [v for v in "xy" if v not in names]
    return expr.translate(str.maketrans(dict(zip(sorted(names - {"x", "y", "z"}), free))))


_kind_cache, SPELLINGS = {}, {}

def term_kind(term):
    if term not in _kind_cache:
        body = re.sub(r"^[+-]?(\d+\*)?", "", term)  # drop sign and coefficient
        if not re.search(r"[a-z]", body):  # 2, (-1), ... are all constants
            kind = "constant"
        else:
            names = sorted(set(re.findall(r"[a-z]", body)))
            tree = nr.parse(body)
            tt = "".join(str(nr.evaluate(tree, dict(zip(names, b))) & 1)
                         for b in itertools.product((0, 1), repeat=len(names)))
            kind = "".join(names) + ":" + tt
        _kind_cache[term] = kind
        SPELLINGS.setdefault(kind, Counter())[body] += 1
    return _kind_cache[term]


def kind_sequence(expr):
    seq = []
    for t in split_terms(normalize_names(expr)):
        k = term_kind(t)
        if k not in seq:  # first occurrence counts
            seq.append(k)
    return seq


train["kinds"] = train.src.map(kind_sequence)
test["kinds"] = test.src.map(kind_sequence)


def label(kind):
    if kind == "constant":
        return "constant"
    s = SPELLINGS[kind].most_common(1)[0][0]
    return s[1:-1] if s.startswith("(") and s.endswith(")") and s.count("(") == 1 else s

ex = train[train.src.str.contains(r"\d$") & (train.src_n_vars == 2)].iloc[0]
print("example:", ex.src, "->", [label(k) for k in ex.kinds])
print(f"{len(SPELLINGS)} distinct term kinds")

## 2 · How fixed is the order?

*Order consistency:* for every pair of term kinds that appear together in at least 30 inputs, the share of inputs
that use the more common of the two orders – averaged over all pairs (weighted by how often they co-occur).
50% = no preferred order, 100% = always the same order.

In [ ]:
def pair_counts(seqs):
    c = Counter()
    for ks in seqs:
        for i, a in enumerate(ks):
            for b in ks[i + 1:]:
                c[(a, b)] += 1
    return c


def consistency(seqs, min_n=30):
    c = pair_counts(seqs)
    total = majority = 0
    for (a, b), n in c.items():
        if a < b:
            m = c.get((b, a), 0)
            if n + m >= min_n:
                total += n + m
                majority += max(n, m)
    return majority / total


rng = random.Random(0)
shuffled = [rng.sample(s, len(s)) for s in train.kinds]
cons = {"training data": consistency(train.kinds), "test data": consistency(test.kinds),
        "random order (baseline)": consistency(shuffled)}

const_last = {}
for name, d in [("training data", train), ("test data", test)]:
    with_const = d.kinds[d.kinds.map(lambda s: "constant" in s)]
    const_last[name] = (with_const.map(lambda s: s[-1] == "constant").mean(), len(with_const))
const_last["random order (baseline)"] = (np.mean([s[-1] == "constant" for s in shuffled if "constant" in s]),
                                         sum("constant" in s for s in shuffled))

fig, (ax, axr) = plt.subplots(1, 2, figsize=(12, 3.6))
names = list(cons)
colors = [C["accent"], C["soft"], C["neutral"]]
ax.barh(names, [cons[n] for n in names], color=colors, height=0.6)
for y, n in enumerate(names):
    ax.text(cons[n] + 0.01, y, pct(cons[n]), va="center", fontweight="bold")
ax.set_xlim(0.4, 1.05)
ax.invert_yaxis()
ax.xaxis.set_major_formatter(lambda v, _: pct(v))
ax.set_title("Pairs of terms written in their usual order")

axr.barh(names, [const_last[n][0] for n in names], color=colors, height=0.6)
for y, n in enumerate(names):
    axr.text(const_last[n][0] + 0.01, y, f"{pct(const_last[n][0])}  (n={const_last[n][1]:,})", va="center", fontweight="bold")
axr.set_xlim(0, 1.25)
axr.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
axr.invert_yaxis()
axr.set_yticks([])
axr.xaxis.set_major_formatter(lambda v, _: pct(v))
axr.set_title("Inputs with a constant: constant is the last term")

fixed = cons["training data"] > 0.7
headline(fig, "The training data writes terms in a largely fixed order" if fixed else "Terms appear in no fixed order",
         f"Order consistency {pct(cons['training data'])} vs. {pct(cons['random order (baseline)'])} for random order; "
         f"the constant is always the last term. Test data follows the same order (same generator).")
save(fig, "13_term_order")

## 3 · Which terms come first?

The 10 most frequent term kinds in the training data. Each cell: in what share of inputs containing both, the **row**
term comes **before** the column term. Blue = row usually first, orange = row usually second, white = no preference.

In [ ]:
c = pair_counts(train.kinds)
freq = Counter(k for s in train.kinds for k in s)
top = [k for k, _ in freq.most_common(11) if k != "constant"][:9] + ["constant"]
grid = np.full((len(top), len(top)), np.nan)
for i, a in enumerate(top):
    for j, b in enumerate(top):
        if i != j:
            n, m = c.get((a, b), 0), c.get((b, a), 0)
            if n + m >= 30:
                grid[i, j] = n / (n + m)

# sort rows/columns by how often a term comes first overall, so the order reads top-left to bottom-right
first_score = np.nanmean(grid, axis=1)
idx = np.argsort(-first_score)
grid, top = grid[np.ix_(idx, idx)], [top[i] for i in idx]

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(grid, cmap=BEFORE_AFTER, vmin=0, vmax=1)
for i in range(len(top)):
    for j in range(len(top)):
        if not np.isnan(grid[i, j]):
            ax.text(j, i, pct(grid[i, j]), ha="center", va="center", fontsize=8.5,
                    color="white" if grid[i, j] > 0.85 or grid[i, j] < 0.15 else "#222222")
labels = [label(k) for k in top]
ax.set_xticks(range(len(top)), labels, rotation=40, ha="right", family="monospace")
ax.set_yticks(range(len(top)), labels, family="monospace")
ax.set_xlabel("… comes before this term")
ax.set_ylabel("share of inputs where this term …")
ax.grid(False)
cb = fig.colorbar(im, ax=ax, fraction=0.04)
cb.ax.yaxis.set_major_formatter(lambda v, _: pct(v))
strong = np.nanmean((grid >= 0.9) | (grid <= 0.1))
headline(fig, "Many pairs of terms almost always appear in the same order – and the constant always comes last",
         f"{pct(strong)} of these pairs are in the same order in at least 90% of inputs. Sorted from 'usually first' to 'usually last'.")
save(fig, "14_term_precedence")

## 4 · Does breaking the usual order break the model?

For every reordered input from `05` (inputs the model originally solved): the share of its term pairs that are now in
the **unusual** order (only pairs with a clear preference in training, ≥ 75%). If the model relies on the training
order, it should fail more often the more the order is violated – and when the constant is no longer last.

In [ ]:
direction = {}
for (a, b), n in c.items():
    m = c.get((b, a), 0)
    if n + m >= 50 and n / (n + m) >= 0.75:
        direction[(a, b)] = True  # usual: a before b


def violation(expr):
    ks = kind_sequence(expr)
    pairs = [(a, b) for i, a in enumerate(ks) for b in ks[i + 1:]]
    judged = [(b, a) in direction for a, b in pairs if (a, b) in direction or (b, a) in direction]
    return np.mean(judged) if judged else np.nan

ro = probes[(probes.probe == "reorder terms") & (probes.orig == "correct")].copy()
ro["violation"] = ro.new_src.map(violation)
ro["const_last"] = ro.new_src.map(lambda s: kind_sequence(s)[-1] == "constant")
ro["has_const"] = ro.new_src.map(lambda s: "constant" in kind_sequence(s))
test["violation"] = test.src.map(violation)

bins, blabels = [-0.001, 0.1, 0.25, 0.5, 1.0], ["0-10%", "10-25%", "25-50%", "over 50%"]
t = (ro.dropna(subset=["violation"]).assign(b=lambda d: pd.cut(d.violation, bins, labels=blabels))
       .groupby("b", observed=True).ok.agg(n="size", k="sum"))
t["rate"] = t.k / t.n
t["lo"], t["hi"] = wilson(t.k, t.n)

fig, (ax, axr) = plt.subplots(1, 2, figsize=(12, 3.9), gridspec_kw={"width_ratios": [1.6, 1]})
x = np.arange(len(t))
ax.bar(x, t.rate, color=C["accent"], width=0.6)
ax.errorbar(x, t.rate, yerr=[t.rate - t.lo, t.hi - t.rate], fmt="none", ecolor="black", capsize=3, lw=1)
for xi, (r, n, hi) in enumerate(zip(t.rate, t.n, t.hi)):
    ax.text(xi, hi + 0.01, f"{pct(r)}\nn={n:,}", ha="center", fontsize=8.5)
ax.set_xticks(x, t.index)
ax.set_xlabel("share of term pairs in the unusual order after reordering")
ax.set_ylabel("still correct")
ax.yaxis.set_major_formatter(lambda v, _: pct(v))
ax.set_ylim(0, max(0.3, t.hi.max() + 0.08))
ax.set_title("More violated order → fewer correct answers?")

cc = ro[ro.has_const].groupby("const_last").ok.agg(n="size", k="sum").reindex([True, False])
cc["rate"] = cc.k / cc.n
cc["lo"], cc["hi"] = wilson(cc.k, cc.n)
names = ["constant still last", "constant moved"]
axr.bar(names, cc.rate, color=[C["accent"], C["warm"]], width=0.55)
axr.errorbar(names, cc.rate, yerr=[cc.rate - cc.lo, cc.hi - cc.rate], fmt="none", ecolor="black", capsize=3, lw=1)
for xi, (r, n, hi) in enumerate(zip(cc.rate, cc.n, cc.hi)):
    axr.text(xi, hi + 0.01, f"{pct(r)}\nn={n:,}", ha="center", fontsize=8.5)
axr.yaxis.set_major_formatter(lambda v, _: pct(v))
axr.set_ylim(0, max(0.3, cc.hi.max() + 0.08))
axr.set_title("Reordered inputs with a constant")

monotone = bool(np.all(np.diff(t.rate.to_numpy()) <= 0.02))
headline(fig, "The more the usual order is broken, the fewer answers survive" if monotone
         else "Breaking the usual order breaks answers – but not in proportion",
         f"From {pct(t.rate.iloc[0])} still correct (few pairs out of order) to {pct(t.rate.iloc[-1])} (most pairs); "
         f"constant moved away from the end: {pct(cc.rate[False])} vs. {pct(cc.rate[True])}. "
         f"Natural test inputs have {pct(test.violation.mean(), 1)} of pairs out of order.")
save(fig, "15_reorder_vs_convention")

## Summary

* **Training and test data both use a largely fixed term order** – far from random – and the constant is always the
  last term. The generator writes terms in a typical sequence, just as it writes `x` left and `y` right.
* **The model depends on it:** the more term pairs a reordering puts into the unusual order, the fewer answers stay
  correct, and moving the constant away from the end is especially harmful. Even small deviations already cost a lot.
* Together with the operand convention (`07`, figure 11): the model has learned the generator's **writing conventions**,
  not the algebra – which is why equivalent rewrites break it.